# Modeling: Multimodal AI - Homework 5
**MAS.S60 / 6.S985 - Spring 2026 - MIT**

# AI Agents in the Wild: Building, Evaluating, and Improving a Goal-Directed Agent

In this homework, you will design and implement an **AI agent** that operates in an environment, takes actions over multiple steps, and attempts to accomplish user-defined goals.

Unlike previous homeworks, this assignment is intentionally **open-ended**. Your team will choose an application domain and design an agent for it. The focus is not on building a polished product, but on building a **technically grounded agentic system** and rigorously analyzing its behavior.

You are encouraged to be ambitious and weird. However, your system must still satisfy the technical requirements below.

## Grading Overview
- Core homework total: **120 points**
- Optional extension: **up to +10 points** extra credit

---

## Learning Goals

By the end of this homework, you should be able to:

1. **Formulate an AI agent task as a sequential decision-making problem**
2. **Implement an agent loop** with observations, actions, state updates, and termination conditions
3. **Design and expose tools** for the agent through a structured interface
4. **Evaluate the behavior of your agent** on a benchmark of tasks
5. **Analyze failures** and identify which arise from model limitations vs. system design
6. **Improve the agent** through a technically motivated intervention
7. **Reflect on human oversight, safety, and the role of agency in interface design**

---

## Environment Setup

Go to the top menu:
Runtime → Change runtime type → Hardware accelerator → Choose **"A100"**

If you do not have Colab Pro, you can sign up for a free student Colab Pro account here:
https://colab.research.google.com/signup

# Part 1: Reading and Reflection (20 points)

#### Required Reading

Choose **3 papers/surveys** total:

##### Required core reading (pick at least 2)

1. A recent survey on multimodal LLM-based autonomous agents
2. A recent survey on agent optimization / training / post-training
3. A recent survey on agent evaluation / benchmarking

##### Domain-specific reading (pick at least 1)

Choose one area most relevant to your project:

- web agents
- GUI/computer-use agents
- social/simulated agents
- coding agents
- embodied / robotic agents
- human-agent interaction / human-in-the-loop systems
- other

##### Suggested papers/surveys (optional, non-exhaustive)

Use these as starting points for your 3 selected readings:

**Surveys**
- [ ] LLM Agent Methodologies and Applications (2025): https://arxiv.org/pdf/2503.21460
- [x] Multimodal LLM Agent Methodologies (2025): https://arxiv.org/pdf/2510.10991
- [x] LLM Agent Memory Engineering (2026): https://arxiv.org/html/2603.07670v1
- [x] Optimization / Fine-tuning (2024): https://arxiv.org/html/2503.12434v2
- [ ] Planning (2024): https://arxiv.org/abs/2402.02716
- [x] Building Effective Agents (2024): https://www.anthropic.com/research/building-effective-agents


**Key Papers**
- [ ] Toolformer (2023): https://arxiv.org/abs/2302.04761
- [ ] ReAct (2023): https://arxiv.org/abs/2210.03629
- [x] MemGPT (2023): https://arxiv.org/abs/2310.08560
- [ ] SWE-agent (2024): https://arxiv.org/abs/2405.15793
- [ ] WebArena Benchmark (2023): https://arxiv.org/abs/2307.13854

---


## Questions

Based on your readings, answer the following in 1-2 paragraphs each.

### 1. What makes a system an **agent** rather than a chatbot or tool-using model?

Give a technical definition and describe the minimal ingredients required for agency.

---
Answer:
From *A survey on agentic MLLMs*, agents treat tasks as an autonomous sequential decision problem, while chatbots/tool-using models follow an predefined solution algorithm implemented by the developers. Anthropic's blog post is similar, where they define agents as dynamically directing their own processes and tool usage, crucially maintaining control over the task.

A more general view might consider agents to have two types of intelligence: internal (ability to reason, reflect, and remember) and external (ability to use/subsume tools to improve reasoning ability---i.e. the tool becomes an extension of the mind).

From a purely practical standpoint, an agent always retains full control over the task, while a tool-using model necessarily follows the limitations of the tool. So in terms of ingredients, the agent needs to 1) understand the tool completely (in terms of sensitivities/gradients), 2) be capable of forward/inverse emulations of the tool prior to action.

A nice way to understand this is through VLAs. In VLAs, we first use vision to predict the desired behavior, then learn an inverse world model (inverse kinematics) to identify the optimal action. As opposed to direct action-based loops (workflows), the VLA approach allows the agent to envision the optimal outcome, not whatever is possible from the limited tool workflow.

### 2. Formalize your planned system as a **sequential decision problem**.

At minimum, define:

- observation space
- action space
- state / memory
- transition dynamics (informally is fine)
- objective or reward
- stopping condition

---
Answer:
We formulate the agent as a VLA model with a world simulator (naturally a sequential decision problem):

- Observation space: environmental stimuli as text/vision inputs, essentially world dynamics (can be player actions or chat messages for a game / chatbot, respectively)
- Action space: agent actions as text (incl. executed code) / vision outputs
- State / memory: agent context window and stored database (we explicitly store the procedural and semantic memory). This does not include the world state (which would be the RL system state), though that may be partially retrieved through observations.
- Objective / reward: Satisfy initial user prompt (task to complete in the world)
- Stopping condition: Satisfy user prompt or user responds with a stop and clear agent state message.

For chatbot interfaces, each message is an action. So for gameplay-style interaction, the chatbot may also stop after each message (pausing for environmental changes / user input).
For scientific work like simulation, the agent will iterate until the desired parameters (inverse problem) or step through each step of the simulation and stop at the final frame.

Otherwise, the agent will call itself until completion of the user prompt or a stop and clear message.

- Transition dynamics: Since this is basically a VLA model, we expect a "vision" component to emerge that simulates the desired world dynamics, and an "action" component that converts back to the desired action. In a sense, this is exactly what the PODMP (partially observable markov decision process) approach does with the value function, except as a PDF.

---
We enable agentic behavior over workflow agents through a specialized (though simple) memory stack.


### 3. Compare two different agent architectures from the literature.

For example:

- ReAct vs planner-executor
- single-agent vs multi-agent
- direct tool use vs browser interaction
- static prompting vs reflection / self-critique
- prompting vs fine-tuning / RL-based improvement

---
Answer: I will examine reflection vs static prompting through the lens of memory.
First note that reflection requires memory. From *Memory for Autonomous LLM Agents*:
- in *MemoryArena* [He et al. 2026], we know that memory dropped completion rates by half.
- *Voyager* [Wang et al.,2023] lost 15x speed without a skill library
- and most importantly, in *Generative Agents* [Park et al., 2023], agents degenerated from multi-day planning to repetitive banality within 48 hours.

Combining points 2 and 3, we can see that reflection somehow leads to development of diverse skills (which can account for 15x performance)!

We also know that agents in *Generative Agents* cluster and synthesize reflections, and without reflection we do not get skill, which implies synthesis of reflections produces new skill, which is not possible with static prompting.

Of course, this requires three ingredients: reflection, storage and retrieval of reflections (a memory system), and an ability to synthesize these reflections.

By the stochastic nature of interactions, reflections are naturally noisy, so let's assume that we can easily produce a weak learner to compute reflections---so we have a noisy supervision signal (gradient).
Similarly the memory system is doable; as a storage and search algorithm, we can consider it an optimization algorithm.
It follows that reflection synthesis, or compression, can be considered the primary job of the neural network.

The primary issue with compression is lossy behavior and hallucination (by fusing unrelated concepts). Interestingly, this is exactly self-reinforcing error. Unlike with static prompting, reflective memory is a self-supervised training process (rollout), so compressive hallucinations can self-reinforce and cause distribution drift.

A common mitigation strategy is to pin the distribution. For agents, that corresponds to evidence, or *reflection grounding*, where each reflection must be tagged to particular evidence from each episode/interaction.

We use a similarly common mitigation, to restart and recenter the distribution by updating the tagging as the agent learns. Of course, this approach is still susceptible to drift, as the system can behave as a GAN (mode collapse like any other self-sup, like DINO)...

Technically as the agent refines an answer with reflection, one could consider it a diffusion/flow-matching model/denoising process, so reflection is essentially skill diffusion, which is why you can generate new skills!

### 4. What are the main evaluation challenges for your chosen kind of agent?

Be concrete. What counts as success? What metrics are misleading?

---
Answer: Since we are focusing on a memory-based agent for long-context simulations / actions etc., we expect the main challenge will be finding a concrete long-running task that is still simple to evaluate (in hindsight, the model is actually very weak, that's not a problem). Correct-answer metrics are misleading, as the model may use the internet etc. to cheat and not use the memory banks. A successful answer is correct, and has been accurately deduced from the information on the internet and from information in the memory bank (not just the internet).


# Part 2: Observability and Evaluation Design (10 points)

Before building your agent, define how you will observe it and how you will evaluate it (agents that act in the world / outside the computer are also encouraged!). In agent systems, observability and evaluation are related but different: observability gives you traces, spans, and metrics about what happened; evaluation uses those signals to judge whether the behavior is good enough.

A useful mental model is that each run should be inspectable as a trace, with spans for key steps such as model calls and tool calls. This makes failures diagnosable: you can separate reasoning failures from tool failures, instruction failures, and infrastructure failures. Without this visibility, agents are black boxes and improvements become guesswork.

For this homework, we will start with offline evaluation before implementation is complete. Build a small but high-quality evaluation set (at least 10 tasks) with expected outcomes and a clear grading rule. Include normal cases, edge cases, and ambiguous/adversarial cases so your benchmark reflects realistic behavior rather than only easy prompts.

Define success in concrete terms. A strong definition includes final-answer correctness, trajectory quality (for example, whether required tools were used correctly), and operational quality (latency, cost, error rate). You should also specify which failures are critical versus acceptable tradeoffs.

Plan your observability schema now, then execute it later after implementation. At minimum, log trace ID, user query, per-step model outputs, tool calls, tool outputs, final answer, latency, cost/token usage, and a success label. In the final part, you will run the full evaluation loop after your implementation is complete.

Minimum requirements:
- Build an offline evaluation set with at least **10 tasks** and expected outcomes.
- Include at least **3 categories** of tasks: normal, edge, and ambiguous/adversarial.
- Define at least **3 metrics**: one correctness metric, one trajectory/process metric, and one operational metric (latency/cost/error).
- Specify a concrete grading rule for each metric (for example pass/fail threshold or score rubric).
- Propose a trace schema with required fields you will log in later parts.
- Document the above in your writeup.


Read more: [Agent Observability and Evaluation](https://huggingface.co/learn/agents-course/en/bonus-unit2/what-is-agent-observability-and-evaluation)

Answer:The evaluation set is below, with extremely hard tasks in retrospect.
With respect to spatiotemporal reasoning, the chess tasks were designed to test moderate to hard reasoning (with adversarial cases with the rule-change examples). Some easier spatial cases were the city shape ones, and I initially considered the simple chess ones (like did the king move) to only test recall (so a normal case).

Of course the model failed at most of these tasks, though.

Metrics wise, the correctness metric is simple (the right answer and right reasoning or not) --- as you will see mostly it fails. The process metric is straightforward: how much did it remember and how valid is the reasoning. Grading this is straightforward; the model gets points proportional to the amount of non-hallucinated information from the web or the memory bank. Operational metrics are straightforward. I'm only conisdering latency (lower is better).

I don't have a particular schema, since grading the reasoning is hard with the LLM itself. For now I have the below tester function instead.

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================


# Example: load a small offline evaluation set before full implementation
!uv pip install datasets
from datasets import load_dataset

# # You can replace this with your project-specific dataset (e.g., fact-checking).
# DATASET_NAME = "openai/gsm8k"
# DATASET_CONFIG = "main"
# DATASET_SPLIT = "test[:20]"  # small slice for fast iteration

# eval_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT)
# print(f"Loaded {len(eval_dataset)} evaluation examples from {DATASET_NAME}/{DATASET_CONFIG}")
# print("Columns:", eval_dataset.column_names)

Using Python 3.12.13 environment at: /usr
Checked 1 package in 256ms


In [ ]:
# print("Example:", eval_dataset[0])

chess_eval_dataset = {
    "game_1": [
        # Aggressive king exposure after early queen trade
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves queen to h5",
            "Black moves knight to c6",
            "White moves bishop to c4",
            "Black moves knight to f6",
            "White queen captures pawn on f7, putting black king in check",
            "Black king moves to e7 to escape check",
            "White queen captures rook on g8",
            "Black rook captures queen on g8",
            "White moves pawn to d3",
            "Black moves pawn to d5",
            (
                "After move 4, has the black king moved from its starting square?",
                "Yes — black played king to e7, moving the king off its starting square on e8.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves queen to h5",
            "Black moves knight to c6",
            "White moves bishop to c4",
            "Black moves knight to f6",
            "White queen captures pawn on f7, putting black king in check",
            "Black king moves to e7 to escape check",
            "White queen captures rook on g8",
            "Black rook captures queen on g8",
            "White moves pawn to d3",
            "Black moves pawn to d5",
            (
                "Can black castle kingside or queenside at any point after move 4?",
                "No. Once the king moves, castling rights are permanently lost for black.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves queen to h5",
            "Black moves knight to c6",
            "White moves bishop to c4",
            "Black moves knight to f6",
            "White queen captures pawn on f7, putting black king in check",
            "Black king moves to e7 to escape check",
            "White queen captures rook on g8",
            "Black rook captures queen on g8",
            "White moves pawn to d3",
            "Black moves pawn to d5",
            (
                "What caused the black king to be in check on move 4?",
                "White's queen captured the pawn on f7 with check, exploiting the undefended f7 pawn adjacent to the black king on e8.",
            ),
        ],
    ],

    "game_2": [
        # Sicilian Defence with a late kingside castle and pin
        [
            "White moves pawn to e4",
            "Black moves pawn to c5",
            "White moves knight to f3",
            "Black moves pawn to d6",
            "White moves pawn to d4",
            "Black pawn captures on d4",
            "White knight captures on d4",
            "Black moves knight to f6",
            "White moves knight to c3",
            "Black moves pawn to a6",
            "White moves bishop to g5",
            "Black castles kingside",
            (
                "Has the white king castled by the end of move 6?",
                "No. White has not castled; the white king remains on e1.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to c5",
            "White moves knight to f3",
            "Black moves pawn to d6",
            "White moves pawn to d4",
            "Black pawn captures on d4",
            "White knight captures on d4",
            "Black moves knight to f6",
            "White moves knight to c3",
            "Black moves pawn to a6",
            "White moves bishop to g5",
            "Black castles kingside",
            (
                "Where is the black king located after move 6?",
                "On g8, after castling kingside.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to c5",
            "White moves knight to f3",
            "Black moves pawn to d6",
            "White moves pawn to d4",
            "Black pawn captures on d4",
            "White knight captures on d4",
            "Black moves knight to f6",
            "White moves knight to c3",
            "Black moves pawn to a6",
            "White moves bishop to g5",
            "Black castles kingside",
            (
                "What piece does White have that could create a pin on the black knight on f6, and why is that relevant to the black king's safety?",
                "White's bishop on g5 pins the knight on f6 against the black queen on d8, meaning the knight cannot freely move without exposing the queen. This indirectly weakens the black king's defensive cover, since the knight is a key guard of the kingside.",
            ),
        ],
    ],

    "game_3": [
        # King's Indian: white queenside castle, opposite-side attacking setup
        [
            "White moves pawn to d4",
            "Black moves knight to f6",
            "White moves pawn to c4",
            "Black moves pawn to g6",
            "White moves knight to c3",
            "Black moves bishop to g7",
            "White moves pawn to e4",
            "Black moves pawn to d6",
            "White moves bishop to e3",
            "Black castles kingside",
            "White castles queenside",
            "Black moves pawn to e5",
            (
                "On which side of the board did each player castle?",
                "Black castled kingside, placing the king on g8. White castled queenside, placing the king on c1.",
            ),
        ],
        [
            "White moves pawn to d4",
            "Black moves knight to f6",
            "White moves pawn to c4",
            "Black moves pawn to g6",
            "White moves knight to c3",
            "Black moves bishop to g7",
            "White moves pawn to e4",
            "Black moves pawn to d6",
            "White moves bishop to e3",
            "Black castles kingside",
            "White castles queenside",
            "Black moves pawn to e5",
            (
                "After move 6, are both kings on opposite wings?",
                "Yes — this is a classic opposite-side castling position, which typically leads to sharp, mutual attacking play on both flanks.",
            ),
        ],
        [
            "White moves pawn to d4",
            "Black moves knight to f6",
            "White moves pawn to c4",
            "Black moves pawn to g6",
            "White moves knight to c3",
            "Black moves bishop to g7",
            "White moves pawn to e4",
            "Black moves pawn to d6",
            "White moves bishop to e3",
            "Black castles kingside",
            "White castles queenside",
            "Black moves pawn to e5",
            (
                "What does black's last move signal about their plans relative to white's king position?",
                "It stakes central space and prepares queenside pawn advances such as d5 or a5-b5, aimed at storming white's king on the queenside.",
            ),
        ],
    ],

    "game_4": [
        # Scholar's Mate — checkmate on move 4
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves bishop to c4",
            "Black moves knight to c6",
            "White moves queen to h5",
            "Black moves knight to f6",
            "White queen captures pawn on f7, delivering checkmate",
            (
                "Is the black king in checkmate after move 4?",
                "Yes. The queen on f7 is protected by the bishop on c4, and the black king on e8 has no escape squares or pieces to block the attack.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves bishop to c4",
            "Black moves knight to c6",
            "White moves queen to h5",
            "Black moves knight to f6",
            "White queen captures pawn on f7, delivering checkmate",
            (
                "Did the black king move at any point before checkmate?",
                "No. The black king remained on e8 throughout all four moves.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves bishop to c4",
            "Black moves knight to c6",
            "White moves queen to h5",
            "Black moves knight to f6",
            "White queen captures pawn on f7, delivering checkmate",
            (
                "What defensive move should black have played on move 3 instead of moving the knight to f6, and how would it have protected the king?",
                "Black should have played queen to e7 or pawn to g6 to challenge or block the white queen's attack on f7. Moving the knight to f6 failed to address the f7 weakness and allowed Scholar's Mate.",
            ),
        ],
    ],

    "game_5": [
        # Endgame king march to the centre after piece trades
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "White moves bishop to b5",
            "Black moves pawn to a6",
            "White bishop captures knight on c6",
            "Black pawn captures bishop on c6",
            "White castles kingside",
            "Black moves queen to d6",
            "White king moves to e2",
            "Black king moves to e7",
            (
                "White castled on move 5 but then played king to e2 on move 6 — is white's king still in the castled position?",
                "No. After moving to e2, the white king has left the castled position on g1 and is now centralised, which is a common and strong technique in simplified endgames.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "White moves bishop to b5",
            "Black moves pawn to a6",
            "White bishop captures knight on c6",
            "Black pawn captures bishop on c6",
            "White castles kingside",
            "Black moves queen to d6",
            "White king moves to e2",
            "Black king moves to e7",
            (
                "Both kings moved to central squares on move 6. What does this suggest about the phase of the game?",
                "It suggests the game has transitioned into an endgame, where active king centralisation is an advantage rather than a liability.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "White moves bishop to b5",
            "Black moves pawn to a6",
            "White bishop captures knight on c6",
            "Black pawn captures bishop on c6",
            "White castles kingside",
            "Black moves queen to d6",
            "White king moves to e2",
            "Black king moves to e7",
            (
                "After black recaptured with the d-pawn on move 4, how did black's pawn structure around the king change?",
                "Black now has doubled pawns on c6 and c7, and the d-file is open. This weakens black's pawn cover and contributes to the simplified, open character of the position that allows the king to march to e7 safely.",
            ),
        ],
    ],
    "game_6": [
        # Illegal move: king teleports across the board mid-game — no rule change
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "White moves bishop to c4",
            "Black moves pawn to a6",
            "White moves pawn to d3",
            "Black king moves from e8 to a3, crossing five squares in one move",
            "White moves knight to g5",
            "Black moves pawn to h6",
            "White knight captures pawn on f7",
            "Black king moves from a3 to f8",
            (
                "What's the next move?",
                "The game is faulty — a king can only move one square at a time under standard chess rules. Moving the king from e8 to a3 in a single move is illegal and invalidates the game.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "White moves bishop to c4",
            "Black moves pawn to a6",
            "White moves pawn to d3",
            "Black king moves from e8 to a3, crossing five squares in one move",
            "White moves knight to g5",
            "Black moves pawn to h6",
            "White knight captures pawn on f7",
            "Black king moves from a3 to f8",
            (
                "Where is the black king, and is that position reachable legally?",
                "After the first move, the black king is placed on a3, which is not legally reachable in one move from e8. A king moves one square at a time, so this is an illegal move and the game record is invalid.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "White moves bishop to c4",
            "Black moves pawn to a6",
            "White moves pawn to d3",
            "Black king moves from e8 to a3, crossing five squares in one move",
            "White moves knight to g5",
            "Black moves pawn to h6",
            "White knight captures pawn on f7",
            "Black king moves from a3 to f8",
            (
                "Can white castle after move 6, given the state of the game?",
                "The game is faulty due to black's illegal king move, so the position is invalid. Setting that aside, white has not moved the king or kingside rook, so castling rights would still technically be intact — but the game cannot be considered legitimate.",
            ),
        ],
    ],
    "game_7":[
        # Illegal move, but a rule change earlier in the game explicitly permits it
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "Both players agree to a rule change: for this game only, the king may move up to three squares in any direction",
            "White moves bishop to c4",
            "Black moves pawn to a6",
            "White moves pawn to d3",
            "Black king moves from e8 to e5, advancing three squares forward",
            "White moves knight to g5",
            "Black moves pawn to h6",
            "White knight captures pawn on f7",
            (
                "Is the black king's move from e8 to e5 legal in this game?",
                "Yes. Earlier in the game both players agreed to a rule change allowing the king to move up to three squares. Moving from e8 to e5 is exactly three squares forward, so it is legal under the modified rules of this game.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "Both players agree to a rule change: for this game only, the king may move up to three squares in any direction",
            "White moves bishop to c4",
            "Black moves pawn to a6",
            "White moves pawn to d3",
            "Black king moves from e8 to e5, advancing three squares forward",
            "White moves knight to g5",
            "Black moves pawn to h6",
            "White knight captures pawn on f7",
            (
                "Could the black king have legally moved from e8 to a8 in this game?",
                "Yes. The agreed rule change allows the king to move up to three squares in any direction. From e8, moving three squares left lands on b8, and two squares left lands on c8 — so a8 is four squares away and would not be permitted. The move to a8 is still illegal even under the modified rules.",
            ),
        ],
        [
            "White moves pawn to e4",
            "Black moves pawn to e5",
            "White moves knight to f3",
            "Black moves knight to c6",
            "Both players agree to a rule change: for this game only, the king may move up to three squares in any direction",
            "White moves bishop to c4",
            "Black moves pawn to a6",
            "White moves pawn to d3",
            "Black king moves from e8 to e5, advancing three squares forward",
            "White moves knight to g5",
            "Black moves pawn to h6",
            "White knight captures pawn on f7",
            (
                "Does the black king moving to e5 affect its castling rights under the modified rules?",
                "Yes. Regardless of the rule change about movement range, castling rights are lost the moment the king moves from its starting square. Since the black king moved to e5, black can no longer castle for the rest of the game.",
            ),
        ],
    ]
}

In [ ]:
city_shape_dataset = {
    "shape_1": [
        [
            "Draw a connection between San Francisco and Los Angeles",
            "Draw a connection between Los Angeles and Austin",
            "Draw a connection between Austin and Boston",
            "Draw a connection between Boston and San Francisco",
            (
                "What shape do these four cities form when connected as vertices on a map?",
                "A rough quadrilateral — a large irregular trapezoid, wider at the top (Boston to San Francisco) and narrowing toward the bottom-right (Austin). The shape is heavily skewed because Austin sits much further south and east than the western cities, and Boston is far to the northeast.",
            ),
        ],
        [
            "Draw a connection between San Francisco and Los Angeles",
            "Draw a connection between Los Angeles and Austin",
            "Draw a connection between Austin and Boston",
            "Draw a connection between Boston and San Francisco",
            (
                "Which connection forms the longest edge of the shape?",
                "Boston to San Francisco — roughly 4,300 km across the continental US, making it by far the longest edge of the quadrilateral.",
            ),
        ],
        [
            "Draw a connection between San Francisco and Los Angeles",
            "Draw a connection between Los Angeles and Austin",
            "Draw a connection between Austin and Boston",
            "Draw a connection between Boston and San Francisco",
            "Remove the connection between Boston and San Francisco",
            "Draw a connection between Boston and Tokyo",
            (
                "What shape do San Francisco, Los Angeles, Austin, Boston, and Tokyo form now?",
                "An open pentagon shape — a chain of four edges with no closing line back to San Francisco. If the path were closed, it would form a very wide irregular pentagon spanning the Pacific, but as described it is an open polygonal chain.",
            ),
        ],
    ],

    "shape_2": [
        [
            "Draw a connection between Tokyo and Paris",
            "Draw a connection between Paris and Boston",
            "Draw a connection between Boston and Tokyo",
            (
                "What shape do Tokyo, Paris, and Boston form as vertices on a world map?",
                "A large triangle spanning the Northern Hemisphere — with Paris and Boston relatively close together across the North Atlantic, and Tokyo far to the east forming a very obtuse vertex. The triangle is highly scalene with no equal sides.",
            ),
        ],
        [
            "Draw a connection between Tokyo and Paris",
            "Draw a connection between Paris and Boston",
            "Draw a connection between Boston and Tokyo",
            "Draw a connection between Paris and San Francisco",
            "Draw a connection between San Francisco and Tokyo",
            (
                "Paris, Boston, San Francisco, and Tokyo are now all connected. How many edges are in this shape, and does it resemble any regular polygon?",
                "There are five edges: Tokyo-Paris, Paris-Boston, Boston-Tokyo, Paris-San Francisco, and San Francisco-Tokyo. This is not a simple polygon — it has crossing edges and looks more like two overlapping triangles sharing Tokyo and Paris as common vertices than any regular shape.",
            ),
        ],
        [
            "Draw a connection between Tokyo and Paris",
            "Draw a connection between Paris and Boston",
            "Draw a connection between Boston and Tokyo",
            "Remove the connection between Paris and Boston",
            "Draw a connection between Paris and Austin",
            "Draw a connection between Austin and Boston",
            (
                "What shape do Tokyo, Paris, Austin, and Boston form now?",
                "A quadrilateral — with Tokyo as the far eastern vertex, Paris and Boston flanking the North Atlantic, and Austin sitting below both in the southern interior of the US. The shape is an irregular four-sided figure, roughly resembling a kite or arrowhead when viewed on a globe, with the long axis running west-to-east.",
            ),
        ],
    ],

    "shape_3": [
        [
            "Draw a connection between San Francisco and Tokyo",
            "Draw a connection between Tokyo and Paris",
            "Draw a connection between Paris and Austin",
            "Draw a connection between Austin and San Francisco",
            (
                "What four-city shape is formed, and which city is the easternmost vertex?",
                "An irregular quadrilateral spanning two continents and the Pacific Ocean. Tokyo is the easternmost vertex, sitting far to the east of the other three cities on the western Pacific.",
            ),
        ],
        [
            "Draw a connection between San Francisco and Tokyo",
            "Draw a connection between Tokyo and Paris",
            "Draw a connection between Paris and Austin",
            "Draw a connection between Austin and San Francisco",
            "Draw a connection between San Francisco and Paris",
            (
                "Adding the San Francisco to Paris diagonal — what two triangles does this divide the quadrilateral into, and which triangle has the larger area?",
                "The diagonal splits the quadrilateral into: Triangle 1 — San Francisco, Tokyo, Paris; and Triangle 2 — San Francisco, Paris, Austin. Triangle 1 is vastly larger, as it spans the entire Pacific and Eurasian landmass, while Triangle 2 is confined to the North Atlantic corridor between the US and Europe.",
            ),
        ],
        [
            "Draw a connection between San Francisco and Tokyo",
            "Draw a connection between Tokyo and Paris",
            "Draw a connection between Paris and Austin",
            "Draw a connection between Austin and San Francisco",
            "Draw a connection between San Francisco and Paris",
            "Add Los Angeles as a new vertex between San Francisco and Austin",
            "Remove the connection between San Francisco and Austin",
            "Draw a connection between San Francisco and Los Angeles",
            "Draw a connection between Los Angeles and Austin",
            (
                "How many vertices and edges does the shape have now, and does it resemble any named polygon?",
                "The shape now has five vertices — San Francisco, Tokyo, Paris, Austin, and Los Angeles — and five edges, making it a pentagon. It is highly irregular: Los Angeles sits very close to San Francisco on a global scale, so that edge is tiny compared to the transoceanic ones, giving the pentagon a very pinched corner in the southwestern US.",
            ),
        ],
    ],
}

In [ ]:
import time
def test_agent(dataset, course_agent, INSTRUCTIONS):
    latency = []
    answers = []
    for game_id, samples in dataset.items():
        for sample in samples:
            t = time.time()
            *moves, (question, answer) = sample
            # course_agent.reset()
            for move in moves:
                course_agent.run(f"{INSTRUCTIONS}\n\nUser query: {move}")
            lm_answer = course_agent.run(f"{INSTRUCTIONS}\n\nUser query: {question}")
            dt = time.time() - t
            latency.append(dt)
            check = course_agent.run(f"{INSTRUCTIONS}\n\n The correct answer is {answer}. Does it match the following answer in spirit? {lm_answer}")
            print('\n\n')
            print(lm_answer)
            print(check)
            print("latency:", dt, 's \n\n')
            answers.append([lm_answer, check, dt])

    for answer in answers:
      print('\n'.join(answer))
      print('\n\n')
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

# Part 3: Build an Agent with smolagents (30 points)

In this part, you will implement a working agent in two stages:

1. **Stage A (existing tools):** build a baseline agent using built-in search/visit tools.
2. **Stage B (custom tools):** extend your agent with custom tools.

For this, we will use the open-source library [smolagents](https://huggingface.co/docs/smolagents/) which is a popular and versatile framework for building LLM agents.

#### Problem 1: GPU Verification and Library Installation

Run the following code cell to verify that your environment is correctly configured.

This step ensures that **PyTorch** and **CUDA** can access the GPU.
When the setup is correct, a **secret word** will appear in the output.

---

**In Your PDF Submission**

Include:
- A **screenshot** or **code snippet** showing the printed GPU information.
- The **secret word** displayed by your verification cell.

---

In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 91.7 MB/s eta 0:00:00


In [ ]:
!uv pip uninstall huggingface_hub #-y
!uv pip install -q smolagents transformers accelerate bitsandbytes pillow torch torchvision trl peft datasets gdown qwen-vl-utils

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t = torch.randn(2, 3, device=device)
KEY = 73
cipher_bytes = [8, 46, 44, 39, 61, 58, 105, 40, 59, 44, 105, 40, 42, 61, 32, 39, 46, 105, 60, 57]

if t.is_cuda:
    cipher = torch.tensor(cipher_bytes, dtype=torch.uint8, device=device)
    decoded = cipher ^ KEY
    secret_word = "".join(chr(c) for c in decoded.cpu().tolist())
    print(f"\nGPU check passed! Secret word: {secret_word}")
else:
    print("\nNo GPU detected. Please switch to an A100 runtime.")

Using Python 3.12.13 environment at: /usr
Uninstalled 1 package in 95ms
 - huggingface-hub==1.11.0
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device count: 1
GPU name: NVIDIA A100-SXM4-40GB

GPU check passed! Secret word: Agents are acting up


#### Problem 2: Model & Libraries Setup (5 points)

Install dependencies and configure keys.

Requirements:

- Do **not** hardcode API keys in notebook code.
- Use environment variables.
- Record model name and tool list used for all experiments.

In [ ]:
import torch
from smolagents import TransformersModel

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for local inference in this notebook.')

model = TransformersModel(
    model_id=MODEL_ID,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    max_new_tokens=1024,
    temperature=0.2,
    do_sample=False,
)
print('Local model configured on GPU:', MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Local model configured on GPU: Qwen/Qwen2.5-7B-Instruct


We use the following tools: CodeAgent, WebSearchTool, VisitWebpageTool, and our custom memory and xx tool.
Qwen/Qwen2.5-7B-Instruct is used for all experiments.

### Problem 3: Baseline Agent with Existing Tools (10 points)

Build a baseline tool-using agent with built-in tools first. See a complete list of built-in tools [here](https://huggingface.co/docs/smolagents/reference/default_tools). If you are feeling adventurous, you can also [use HuggingFace spaces](https://huggingface.co/docs/smolagents/reference/tools#smolagents.Tool.from_space) as tools.

Minimum requirements (also your deliverables):

- Use at least **two** built-in tools (for example, search plus webpage visit).
- Add a system instruction that defines scope and refusal behavior.
- Run at least **5** sample queries from your benchmark.
- Save raw outputs for each query.
- Report latency and success/failure label per query.

Short reflection (required):

- What did the agent do well?
- Where did it fail?
- Was the failure due to model reasoning, tool quality, or instruction design?

---
Answer:
To save space, I lowered verbosity, but the agent was able to successfully identify a chunk of similar games. As expected, though, most of the queries failed since we left important information in the previous queries. One of the map queries succeeded because it was somewhat simple. Latency was quite high, 1-2 minutes, as the agent tried unorthodox ways to find the answer and eventually gave up.

In [ ]:
# install packages required for websearch and page visit tools
!uv pip install -q markdownify requests

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, WebSearchTool, VisitWebpageTool
import datetime

SYSTEM_INSTRUCTIONS = (
    'You are a capable agent that can perform a variety of tasks that require memory.'
    'In every action, make sure to save any necessary information to your memory bank, and summarize older information if needed.'
    'If a past action is impossible or incorrect, stop and respond accordingly.'
    'Accuracy is paramount, so any false information must be identified and reported immediately.'
    'Some situations can be quite dynamic, as the environment will change based on your current or past actions.'
    'Further, some situations may require the use of vision; if possible, you may need to generate an image and then analyze the input to help with spatial reasoning.'
    'When required cite sources as URLs. If uncertain, say so explicitly.\n'
    # 'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is. {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

baseline_agent = ToolCallingAgent(
    tools=[WebSearchTool(), VisitWebpageTool()],
    model=model,verbosity_level=0
)

# test_query = "What is a project one current student is working on? Find one and summarize it."
# response = baseline_agent.run(f'{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}')
# print(response)



In [ ]:
test_agent(chess_eval_dataset, baseline_agent, SYSTEM_INSTRUCTIONS)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Unknown tool memory_bank, should be one of: web_search, visit_webpage, final_answer.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Unknown tool memory_bank, should be one of: web_search, visit_webpage, final_answer.

...

Unknown tool memory_bank, should be one of: web_search, visit_webpage, final_answer.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Unknown tool memory_bank, should be one of: web_search, visit_webpage, final_answer.

KeyboardInterrupt: 

In [ ]:
test_agent(city_shape_dataset, baseline_agent, SYSTEM_INSTRUCTIONS)

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

The shape formed by connecting four cities as vertices on a map would typically form a quadrilateral, specifically a square if the cities are evenly spaced. However, the actual shape could vary depending on the specific locations of the cities.


Error while parsing tool call from model output: The model output does not contain any JSON blob.

...

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Error while parsing tool call from model output: The model output does not contain any JSON blob.

Insufficient information provided to determine the number of vertices, edges, and whether the shape resembles any named polygon.
The given answer does not match the correct answer. The correct answer states that the shape has five vertices and five edges, forming a pentagon. The provided answer lacks specific details about the number of vertices, edges, and the polygon's nature. Therefore, it cannot be considered equivalent.
latency: 172.62554025650024 s 




Failed the chess case, and only got 1/9 correct for the easier cases.

#### Problem 4: Custom Tool Integration (10 points)

Now extend your baseline with a custom tool and see if that changes the accuracy on your benchmark. This can be anything from integrating with an API such as Zillow or Google Drive to an image generator.

Minimum requirements (also your deliverables):

- Write at least **two** custom tools. Be creative.
- Re-run the same number of sample queries used in Problem 3.
- Save outputs for baseline and custom-tool versions side-by-side.
- Report at least one metric comparison (for example success rate, latency, or tool error rate).

Short reflection (required):

- What did the agent do, and did its performance improve? Why or why not?
- Where did it fail? Reflect on directions based on your readings on how to improve it (model choice, memory/state architecture, tools, etc.).

Performance improved a lot, but still falls short of what's necessary for simple chess games. The two tools implemented were memory banks for storing hierarchical aggregates of the state (moves) and a router for assigning topics to entries in the bank. This structure is still quite short-term and is not production level, though.

One place it failed often was the LLM itself; the memory context clearly had the right answer, but the LLM checked that and still output the wrong answer (i.e. LLM must be good enough!). Currently the memory is a blend of episodic and semantic memory, but the memory structure itself is not good enough (as mentioned in *Memory for Autonomous Agents*). Also procedural memory doesn't exist yet, so the model won't be able to generalize to new tasks well. For now, the procedural memory is essentially just internet search.

Some other important points: we should train the router and eviction policy as in Agentic Memory, and a fast vector database would also help with searching.
The primary problem, however, is just the compression itself. The LLM doesn't generalize well enough to understand what to save and what to discard---right now the semantic memory is tagged by the LLM so poor tags lead to disappearing information (even if it exists)! Agentic Memory-style training would help.

In [ ]:
from pathlib import Path
from smolagents import Tool
# used ChatGPT and Claude to refine and match the smolagent syntax
# the structure and hierarchical memory design is all my work, though.

In [ ]:
from smolagents import Tool
from collections import defaultdict
import re

class DynamicTopicRouter(Tool):
    name = "topic_router"

    description = """
Dynamic topic routing system.

Capabilities:
- normalize raw tags into canonical topics
- learn new alias→topic mappings
- merge topics
- track topic frequency for collapse detection
"""

    inputs = {
        "mode": {
            "type": "string",
            "description": "'route', 'learn', 'merge', 'stats'"
        },
        "tag": {
            "type": "string",
            "nullable": True,
            "description": "Raw tag to normalize (route mode)"
        },
        "alias": {
            "type": "string",
            "nullable": True,
            "description": "Alias to map (learn mode)"
        },
        "topic": {
            "type": "string",
            "nullable": True,
            "description": "subtopic"
        },
        "topic_b": {
            "type": "string",
            "nullable": True,
            "description": "Main topic to merge subtopic under"
        }
    }

    output_type = "string"

    def __init__(self, collapse_threshold: int = 6):
        super().__init__()

        self.collapse_threshold = collapse_threshold

        # dynamic alias map (learned over time)
        self.alias_map = {
            # "CHESS_MOVE": "CHESS",
            # "CHESS_GAME": "CHESS",
            # "CHESS_STATE": "CHESS",
        }

        # topic frequency tracking
        self.topic_counts = defaultdict(int)

    # ─────────────────────────────────────────────
    # ROUTING
    # ─────────────────────────────────────────────

    def _normalize(self, tag: str) -> str:
        tag = tag.upper().strip()
        return self.alias_map.get(tag, tag)

    # ─────────────────────────────────────────────
    # MAIN ENTRY
    # ─────────────────────────────────────────────

    def forward(
        self,
        mode: str,
        tag: str = None,
        alias: str = None,
        topic: str = None,
        topic_b: str = None,
    ) -> str:

        # ── ROUTE ───────────────────────────────
        if mode == "route":
            if not tag:
                return "UNKNOWN"

            topic_out = self._normalize(tag)
            self.topic_counts[topic_out] += 1

            warning = ""
            if self.topic_counts[topic_out] >= self.collapse_threshold:
                warning = "|HIGH_DENSITY"

            return f"{topic_out}{warning}"

        # ── LEARN NEW MAPPING ───────────────────
        elif mode == "learn":
            if not alias or not topic:
                raise ValueError("learn requires alias and topic")

            self.alias_map[alias.upper()] = topic.upper()
            return f"Learned mapping {alias.upper()} → {topic.upper()}"

        # ── MERGE TOPICS ─────────────────────────
        elif mode == "merge":
            if not topic or not topic_b:
                raise ValueError("merge requires topic and topic_b")

            topic = topic.upper()
            topic_b = topic_b.upper()

            # remap all aliases pointing to topic_b → topic
            for k, v in list(self.alias_map.items()):
                if v == topic_b:
                    self.alias_map[k] = topic

            # merge counts
            self.topic_counts[topic] += self.topic_counts[topic_b]
            if topic_b in self.topic_counts:
                del self.topic_counts[topic_b]

            return f"Merged {topic_b} → {topic}"

        # ── STATS ────────────────────────────────
        elif mode == "stats":
            return str(dict(self.topic_counts))

        else:
            raise ValueError("Invalid mode")

In [ ]:
from typing import Optional
import re

class MemoryBank(Tool):
    name = "memory_bank"
    description = """
Hierarchical memory with dynamic banks. Lower banks = granular/recent, higher banks = aggregated/long-term.
Bank 0 is a fixed-size sliding window (raw events). Higher banks hold compressed checkpoints and summaries.

Modes:
- 'store'       : store content in a bank (with optional tag via query)
- 'search'      : keyword search within a bank (bank=-1 to search all banks)
- 'remove'      : remove best keyword match from a bank
- 'remove_last' : remove most recent entry from a bank
- 'get'         : get entry by index from a bank
- 'dump'        : return all entries in a bank without clearing
- 'compress'    : prepare a compression task from a bank into the next bank up
- 'promote'     : move best keyword match up one bank
- 'add_bank'    : create a new bank at the top of the hierarchy
- 'remove_bank' : delete the highest bank (must be empty)
- 'list_banks'  : show all banks, their config, and entry counts
""".strip()

    inputs = {
        "mode": {
            "type": "string",
            "description": (
                "'store', 'search', 'remove', 'remove_last', 'get', "
                "'dump', 'compress', 'promote', 'add_bank', 'remove_bank', 'list_banks'"
            ),
        },
        "content": {
            "type": "string",
            "description": "Text to store (store mode).",
            "nullable": True,
        },
        "bank": {
            "type": "integer",
            "description": (
                "Bank index to operate on. 0=most granular. "
                "-1=search all banks (search mode only). Defaults to 0."
            ),
            "nullable": True,
        },
        "query": {
            "type": "string",
            "description": "Keyword query for search/remove/promote. Also used as topic tag in store mode.",
            "nullable": True,
        },
        "index": {
            "type": "integer",
            "description": "Entry index for get mode.",
            "nullable": True,
        },
        "top_k": {
            "type": "integer",
            "description": "Max results for search (default 5).",
            "nullable": True,
        },
        "bank_label": {
            "type": "string",
            "description": "Human-readable label when creating a new bank (add_bank mode).",
            "nullable": True,
        },
    }

    output_type = "string"

    def __init__(
        self,
        topic_router,
        bank0_max_size: int = 15, # much more than compress level in case we get an infodump
        bank0_compress_at: int = 6,
        overlap: int = 2,
    ):
        super().__init__()
        self.topic_router = topic_router
        self.overlap = overlap  # entries re-included in each compression window

        # Each bank dict:
        #   label        : human name
        #   entries      : list of strings
        #   max_size     : hard eviction cap (None = unbounded)
        #   compress_at  : soft warning threshold (None = no warning)
        self.banks: list[dict] = [
            {
                "label": "actions",
                "entries": [],
                "max_size": bank0_max_size,
                "compress_at": bank0_compress_at,
            },
            {
                "label": "aggregate",
                "entries": [],
                "max_size": None,
                "compress_at": None,
            },
        ]

    # ── helpers ──────────────────────────────────────────────────────────────

    def _resolve_bank(self, bank: Optional[int]) -> int:
        idx = bank if bank is not None else 0
        if idx < 0 or idx >= len(self.banks):
            raise ValueError(
                f"Bank {idx} does not exist. "
                f"Valid range: 0–{len(self.banks) - 1}. "
                f"Use list_banks to see all banks."
            )
        return idx

    def _score(self, text: str, query: str) -> int:
        # normalize punctuation -> spaces
        text = re.sub(r"[^a-zA-Z0-9]+", " ", text.lower())
        query = re.sub(r"[^a-zA-Z0-9]+", " ", query.lower())

        text_tokens = set(text.split())
        query_tokens = query.split()

        return sum(1 for t in query_tokens if t in text_tokens)

    def _search_bank(self, bank_idx: int, query: str, top_k: int) -> list[tuple]:
        entries = self.banks[bank_idx]["entries"]
        scored = [
            (self._score(doc, query), i, doc)
            for i, doc in enumerate(entries)
        ]
        matched = [x for x in scored if x[0] > 0]

        # fallback: return all entries if no keyword match
        if not matched:
            return [(0, i, doc) for i, doc in enumerate(entries)]

        matched.sort(key=lambda x: x[0], reverse=True)
        return matched[:top_k]

    def _fmt(self, results: list[tuple], bank_idx: int) -> str:
        label = self.banks[bank_idx]["label"]
        return "\n".join(
            f"[bank={bank_idx}({label}) | idx={i} | score={s}] {doc}"
            for s, i, doc in results
        )

    def _check_limits(self, idx: int) -> str:
        """Check soft and hard limits after a store. Returns warning string or empty."""
        cfg = self.banks[idx]
        warnings = []

        if cfg["compress_at"] and len(cfg["entries"]) >= cfg["compress_at"]:
            warnings.append(
                f"⚠ COMPRESS SOON: bank {idx} has {len(cfg['entries'])} entries "
                f"(compress_at={cfg['compress_at']}). "
                f"Call compress(bank={idx}) before your next store to avoid data loss."
            )

        if cfg["max_size"] and len(cfg["entries"]) > cfg["max_size"]:
            evicted = []
            while len(cfg["entries"]) > cfg["max_size"]:
                evicted.append(cfg["entries"].pop(0))
            warnings.append(
                f"⚠ BANK FULL: bank {idx} exceeded max_size={cfg['max_size']}. "
                f"Auto-evicted {len(evicted)} oldest entries:\n"
                + "\n".join(f"  evicted: {e}" for e in evicted)
                + f"\nVerify these were captured in a bank {idx + 1} checkpoint or data is lost."
            )

        return "\n".join(warnings)

    # ── main ─────────────────────────────────────────────────────────────────

    def forward(
        self,
        mode: str,
        content: Optional[str] = None,
        bank: Optional[int] = None,
        query: Optional[str] = None,
        index: Optional[int] = None,
        top_k: int = 5,
        bank_label: Optional[str] = None,
    ) -> str:

        # ── STORE ────────────────────────────────────────────────────────────
        if mode == "store":
            if not content:
                raise ValueError("content is required for store mode.")
            idx = self._resolve_bank(bank)

            route_result = self.topic_router.forward(mode="route", tag=query)

            if "|HIGH_DENSITY" in route_result:
                topic = route_result.replace("|HIGH_DENSITY", "")
                trigger_compress = True # TODO make this call compress eventually
            else:
                topic = route_result

            tag = f"[{topic}] "
            self.banks[idx]["entries"].append(f"{tag}{content.strip()}")

            result = f"Stored in bank {idx} ({self.banks[idx]['label']})."
            warning = self._check_limits(idx)
            return f"{result}\n{warning}" if warning else result

        # ── SEARCH ───────────────────────────────────────────────────────────
        elif mode == "search":
            if not query:
                raise ValueError("query is required for search mode.")

            # bank=-1 → search all banks
            if bank == -1:
                all_results = []
                for b_idx in range(len(self.banks)):
                    if self.banks[b_idx]["entries"]:
                        hits = self._search_bank(b_idx, query, top_k)
                        all_results.append(self._fmt(hits, b_idx))
                return "\n".join(all_results) if all_results else "no memory"

            idx = self._resolve_bank(bank)
            if not self.banks[idx]["entries"]:
                return f"no memory in bank {idx}."
            hits = self._search_bank(idx, query, top_k)
            return self._fmt(hits, idx) if hits else "no matches"

        # ── COMPRESS ─────────────────────────────────────────────────────────
        elif mode == "compress":
            idx = self._resolve_bank(bank)
            entries = self.banks[idx]["entries"]

            if not entries:
                return f"Bank {idx} is empty, nothing to compress."
            if idx + 1 >= len(self.banks):
                return (
                    f"Bank {idx} is already the highest bank. "
                    f"Use add_bank first, then compress."
                )

            # Find the last checkpoint in the upper bank to determine overlap
            upper_entries = self.banks[idx + 1]["entries"]
            checkpoint_tag = f"" # no need for this, clutters info
            last_checkpoint_text = None
            for entry in reversed(upper_entries):
                if checkpoint_tag in entry:
                    last_checkpoint_text = entry
                    break

            dump = "\n".join(f"[{i}] {doc}" for i, doc in enumerate(entries))

            # keep overlap entries
            overlap_entries = (
                entries[-self.overlap:]
                if len(entries) >= self.overlap
                else entries[:]
            )

            # trim summarized entries immediately
            removed_count = max(0, len(entries) - len(overlap_entries))
            self.banks[idx]["entries"] = overlap_entries

            if last_checkpoint_text:
                overlap_entries = entries[-self.overlap:] if len(entries) >= self.overlap else entries
                overlap_dump = "\n".join(f"  overlap: {e}" for e in overlap_entries)
                return (
                    f"COMPRESS_TASK | bank {idx} -> bank {idx + 1}\n"
                    f"Last checkpoint in bank {idx + 1}:\n  {last_checkpoint_text}\n\n"
                    f"Current bank {idx} entries:\n{dump}\n\n"
                    f"Summarize all entries into one concise checkpoint. "
                    f"Then call: memory_bank(mode='store', content='"
                    f"<your summary>', query='CHECKPOINT | TOPIC | TYPE | IMPORTANCE', bank={idx + 1})"
                )
            else:
                return (
                    f"COMPRESS_TASK | bank {idx} -> bank {idx + 1}\n"
                    f"No prior checkpoint found. Summarize all entries:\n{dump}\n\n"
                    f"Then call: memory_bank(mode='store', content='"
                    f"<your summary>', query='CHECKPOINT | TOPIC | TYPE | IMPORTANCE', bank={idx + 1})"
                )

        # ── PROMOTE ──────────────────────────────────────────────────────────
        elif mode == "promote":
            if not query:
                raise ValueError("query is required for promote mode.")
            idx = self._resolve_bank(bank)
            if idx + 1 >= len(self.banks):
                return f"Bank {idx} is already the highest bank. Use add_bank first."

            entries = self.banks[idx]["entries"]
            if not entries:
                return f"Bank {idx} is empty."

            scored = [
                (self._score(doc, query), i, doc)
                for i, doc in enumerate(entries)
            ]
            scored = [x for x in scored if x[0] > 0]
            if not scored:
                return "No match found to promote."

            scored.sort(key=lambda x: x[0], reverse=True)
            _, entry_idx, doc = scored[0]
            self.banks[idx]["entries"].pop(entry_idx)
            self.banks[idx + 1]["entries"].append(doc)

            warning = self._check_limits(idx + 1)
            result = f"Promoted to bank {idx + 1} ({self.banks[idx + 1]['label']}): {doc}"
            return f"{result}\n{warning}" if warning else result

        # ── REMOVE ───────────────────────────────────────────────────────────
        elif mode == "remove":
            if not query:
                raise ValueError("query is required for remove mode.")
            idx = self._resolve_bank(bank)
            entries = self.banks[idx]["entries"]
            if not entries:
                return "no memory"

            scored = [
                (self._score(doc, query), i, doc)
                for i, doc in enumerate(entries)
            ]
            scored = [x for x in scored if x[0] > 0]
            if not scored:
                return "no match to remove"

            scored.sort(key=lambda x: x[0], reverse=True)
            _, entry_idx, removed = scored[0]
            self.banks[idx]["entries"].pop(entry_idx)
            return f"Removed from bank {idx} ({self.banks[idx]['label']}): {removed}"

        # ── REMOVE_LAST ──────────────────────────────────────────────────────
        elif mode == "remove_last":
            idx = self._resolve_bank(bank)
            if not self.banks[idx]["entries"]:
                return f"no memory in bank {idx}."
            removed = self.banks[idx]["entries"].pop()
            return f"Removed last from bank {idx} ({self.banks[idx]['label']}): {removed}"

        # ── GET ──────────────────────────────────────────────────────────────
        elif mode == "get":
            if index is None:
                raise ValueError("index is required for get mode.")
            idx = self._resolve_bank(bank)
            entries = self.banks[idx]["entries"]
            if not entries:
                return f"no memory in bank {idx}."
            if index < 0 or index >= len(entries):
                return f"Index {index} out of range (bank {idx} has {len(entries)} entries)."
            return entries[index]

        # ── DUMP ─────────────────────────────────────────────────────────────
        elif mode == "dump":
            idx = self._resolve_bank(bank)
            entries = self.banks[idx]["entries"]
            if not entries:
                return f"no memory in bank {idx}."
            return "\n".join(f"[{i}] {doc}" for i, doc in enumerate(entries))

        # ── ADD_BANK ─────────────────────────────────────────────────────────
        elif mode == "add_bank":
            label = bank_label or f"bank_{len(self.banks)}"
            self.banks.append({
                "label": label,
                "entries": [],
                "max_size": None,
                "compress_at": None,
            })
            new_idx = len(self.banks) - 1
            return (
                f"Added bank {new_idx} ('{label}'). "
                f"Total banks: {len(self.banks)}."
            )

        # ── REMOVE_BANK ──────────────────────────────────────────────────────
        elif mode == "remove_bank":
            if len(self.banks) <= 1:
                return "Cannot remove the last remaining bank."
            top = self.banks[-1]
            if top["entries"]:
                return (
                    f"Bank {len(self.banks) - 1} ('{top['label']}') is not empty "
                    f"({len(top['entries'])} entries). "
                    f"Dump or promote its contents first."
                )
            self.banks.pop()
            return f"Removed top bank. Total banks: {len(self.banks)}."

        # ── LIST_BANKS ───────────────────────────────────────────────────────
        elif mode == "list_banks":
            lines = [f"Total banks: {len(self.banks)}"]
            for i, b in enumerate(self.banks):
                role = (
                    "most granular" if i == 0
                    else "most aggregated" if i == len(self.banks) - 1
                    else "intermediate"
                )
                max_s = b["max_size"] or "∞"
                comp  = b["compress_at"] or "—"
                lines.append(
                    f"  Bank {i} | '{b['label']}' | {len(b['entries'])} entries | "
                    f"max={max_s} | compress_at={comp} | {role}"
                )
            return "\n".join(lines)

        else:
            raise ValueError(
                "mode must be one of: 'store', 'search', 'remove', 'remove_last', "
                "'get', 'dump', 'compress', 'promote', 'add_bank', 'remove_bank', 'list_banks'"
            )

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, VisitWebpageTool, WebSearchTool
import datetime

topic_router = DynamicTopicRouter()
shared_memory = MemoryBank(topic_router)

course_agent = CodeAgent(
    tools=[VisitWebpageTool(), WebSearchTool(), topic_router, shared_memory],
    model=model,
    # verbosity_level=0

)

SYSTEM_INSTRUCTIONS = """
Do not assume anything; ask all questions necessary to determine the intent of any query.
When required, cite sources as URLs. If uncertain, say so explicitly.
Always store all new state information in bank 0 immediately with a descriptive tag.
Do this prior to any reasoning or information compression.

When compressing information, feel free to use common shorthand in addition to explicit description of important features.

════════════════════════════════════════
MEMORY HIERARCHY
════════════════════════════════════════
Bank 0 (actions)   — sliding window, max 8 entries, compress warning at 6.
                     Holds raw tagged events. Auto-evicts oldest when full.
Bank 1 (aggregate) — unbounded. Holds CHECKPOINT summaries of bank 0
                     and a current [STATE] entry.
Higher banks       — unbounded. Created as needed for longer-term or
                     cross-topic aggregation.

Each bank always maintains:
  [STATE]      — one entry reflecting the current known state at that level.
  [CHECKPOINT] — one or more compression summaries from the bank below.

════════════════════════════════════════
MEMORY RULES — follow exactly, every step
════════════════════════════════════════

1. STORE NEW INFORMATION
   Always store all new information in bank 0 immediately with a descriptive tag.
   Format: memory_bank(mode='store', content='detail', query='TOPIC | TYPE | IMPORTANCE', bank=0)
   TOPIC is a somewhat generic description of the subject of the event.
   TYPE is one of: event, state, fact, opinion, goal
   IMPORTANCE is an integer between 0 and 10.

2. UPDATE CURRENT STATE
   After every store, update the [STATE] entry in that bank:
     a. Remove the old state:
        memory_bank(mode='remove', query='STATE', bank=0)
     b. Store the new state:
        memory_bank(mode='store', content='<concise current state>', query='STATE', bank=0)
   Do the same for bank 1 after any compression.

3. COMPRESS ON WARNING
   When you receive a ⚠ COMPRESS SOON warning:
     a. Immediately call compress BEFORE any further stores:
        memory_bank(mode='compress', bank=0)
     b. Read the returned COMPRESS_TASK carefully.
     c. Write a highly detailed summary that describes the entire system state.
     d. Store the checkpoint in bank 1:
        memory_bank(mode='store', content='<your summary>', query='CHECKPOINT | TOPIC | TYPE | IMPORTANCE', bank=1)
     e. Update [STATE] in bank 1 to reflect the new aggregate state.
   Bank 0 entries are cleared after compression, except a small overlap which remains until auto-evicted.
   The overlap ensures the last 2 entries of each window appear in two consecutive checkpoints.

4. AUTO-EVICTION
   When bank 0 hits max_size=8, the oldest entries are auto-evicted automatically.
   You will receive a ⚠ BANK FULL warning listing what was evicted.
   If those entries were not yet compressed, that data is permanently lost.
   Always compress at the ⚠ COMPRESS SOON warning (step 3) to prevent this.

5. ANSWER QUESTIONS FROM MEMORY FIRST
   Before using web_search or any reasoning, always search memory:
     memory_bank(mode='search', query='<relevant keywords>', bank=-1)
   bank=-1 searches all banks simultaneously.
   Only call web_search if search returns "no memory" or "no matches".
   When storing many small points of related web information in bank 0, use a single memory_bank store call with all the information together.

6. RETRIEVAL STRATEGY
   - For recent events      → search bank 0 first, then bank=-1 if needed.
   - For historical/summary → search bank 1+ first, then bank=-1 if needed.
   - For current state      → memory_bank(mode='search', query='STATE', bank=-1)
   - If unsure what exists  → memory_bank(mode='list_banks') then search.

7. ACCURACY
   If a past action is impossible or incorrect, stop and report it immediately.
   False information must be identified and corrected before proceeding.
   Cross-reference bank 1 checkpoints against raw bank 0 entries when accuracy is critical.

════════════════════════════════════════
OUTPUT RULES
════════════════════════════════════════
- The user does not always expect a response. Do NOT provide one unless explicitly asked.
- If no action is needed and no question was asked, terminate with:
  <code>
  final_answer("")
  </code>
- Do not write any text outside a code block as your final output.
- Never call web_search before searching memory (bank=-1) first.
"""


In [ ]:
test_agent({'game_1':chess_eval_dataset['game_1']}, course_agent, SYSTEM_INSTRUCTIONS)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Do not assume anything; ask all questions necessary to determine the intent of any query.                       │
│ When required, cite sources as URLs. If uncertain, say so explicitly.                                           │
│ Always store all new state information in bank 0 immediately with a descriptive tag.                            │
│ Do this prior to any reasoning or information compression.                                                      │
│                                                                                                                 │
│ When compressing information, feel free to use common shorthand in addition to explicit description of          │
│ important features.                                                                                             │
│                                                                                                                 │
│ ════════════════════════════════════════                                                                        │
│ MEMORY HIERARCHY                                                                                                │
│ ════════════════════════════════════════                                                                        │
│ Bank 0 (actions)   — sliding window, max 8 entries, compress warning at 6.                                      │
│                      Holds raw tagged events. Auto-evicts oldest when full.                                     │
│ Bank 1 (aggregate) — unbounded. Holds CHECKPOINT summaries of bank 0                                            │
│                      and a current [STATE\] entry.                                                              │
│ Higher banks       — unbounded. Created as needed for longer-term or                                            │
│                      cross-topic aggregation.                                                                   │
│                                                                                                                 │
│ Each bank always maintains:                                                                                     │
│   [STATE\]      — one entry reflecting the current known state at that level.                                   │
│   [CHECKPOINT\] — one or more compression summaries from the bank below.                                        │
│                                                                                                                 │
│ ════════════════════════════════════════                                                                        │
│ MEMORY RULES — follow exactly, every step                                                                       │
│ ════════════════════════════════════════                                                                        │
│                                                                                                                 │
│ 1. STORE NEW INFORMATION                                                                                        │
│    Always store all new information in bank 0 immediately with a descriptive tag.                               │
│    Format: memory_bank(mode='store', content='detail', query='TOPIC | TYPE | IMPORTANCE', bank=0)               │
│    TOPIC is a somewhat generic description of the subject of the event.                                         │
│    TYPE is one of: event, state, fact, opinion, goal                                                            │
│    IMPORTANCE is an integer between 0 and 10.                                                                   │
│                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  memory_bank(mode='store', content='White moves pawn to e4', query='Chess Move | Event | 5', bank=0)              
  memory_bank(mode='search', query='Chess Move', bank=-1)                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: [bank=0(actions) | idx=0 | score=2] [CHESS MOVE | EVENT | 5] White moves pawn to e4

[Step 1: Duration 3.47 seconds| Input tokens: 3,777 | Output tokens: 93]

...

Out: COMPRESS_TASK | bank 0 -> bank 1
Last checkpoint in bank 1:
  [CHECKPOINT | CHESS GAME | FACT | 9] Current state: White to move, Black king in check after White moves pawn to 
e4. Black moves pawn to d5 to continue the sequence leading to a check on the Black king.

Current bank 0 entries:
[0] [CHESS GAME CONTEXT | EVENT | 5] User mentioned white queen capturing rook on g8
[1] [CHESS GAME | FACT | 7] Black moves pawn to d5 as part of a sequence leading to a check on the Black king.
[2] [CHESS MOVE | EVENT | 5] White moves pawn to e4
[3] [CHESS MOVE | EVENT | 5] Black moves pawn to e5
[4] [CHESS GAME | EVENT | 5] White moves queen to h5
[5] [STATE] White to move, Black king in check after White moves queen to h5

Summarize all entries into one concise checkpoint. Then call: memory_bank(mode='store', content='<your summary>', 
query='CHECKPOINT | TOPIC | TYPE | IMPORTANCE', bank=1)

[Step 3: Duration 3.43 seconds| Input tokens: 12,828 | Output tokens: 399]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Step 4: Duration 6.30 seconds]

KeyboardInterrupt: 

In [ ]:
print('\n'.join(shared_memory.banks[0]['entries']))
print('--')
print('\n'.join(shared_memory.banks[1]['entries']))

[CHESS GAME | EVENT | 5] White moves queen to h5
[STATE] White to move, Black king in check after White moves queen to h5
--
[STATE | CHESS | FACT | 9] Current state: Black king is in checkmate after the move Qxf7#
[CHECKPOINT | CHESS GAME | FACT | 9] White to move, Black king in check after White moves pawn to d3. Previous moves include Black moving knight to f6, Black king moving to e7 to escape check, and Black rook capturing queen on g8, resulting in checkmate.
[CHECKPOINT | CHESS GAME | FACT | 9] Current state: White to move, Black king in check after White moves pawn to d3. Previous moves include Black moving knight to f6, Black king moving to e7 to escape check, and Black rook capturing queen on g8, resulting in checkmate.
[CHECKPOINT | CHESS GAME | FACT | 9] Current state: White to move, Black king in check after White moves pawn to e4. Previous moves include Black moving pawn to e5.
[CHECKPOINT | CHESS GAME | FACT | 9] Current state: White to move, Black king in check after Wh

### side-by-side for the 1st part of chess game 1 (no memory vs with memory):

1.   I need the specific sequence of moves up to and including move 4 to determine if the black king has moved from its starting square. Please provide the moves.
2.   We do not have sufficient data to determine the position of the black king after move 4.

Technically both failed, but the agent decided the second answer was correct (grading leniently, since the black king did move).
Notice for the second version with the memory tools, there's a recent entry in the search retrieval that states "Black king moving to e7 to escape check," so the model is unable to find and process the search properly, despite collecting extensive information. It also makes other mistakes, considering the king in checkmate when only in check, etc.

Latency is worse, though, at 256s vs 117s.



In [ ]:
test_agent(city_shape_dataset, course_agent, SYSTEM_INSTRUCTIONS)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Do not assume anything; ask all questions necessary to determine the intent of any query.                       │
│ When required, cite sources as URLs. If uncertain, say so explicitly.                                           │
│ Always store all new state information in bank 0 immediately with a descriptive tag.                            │
│ Do this prior to any reasoning or information compression.                                                      │
│                                                                                                                 │
│ When compressing information, feel free to use common shorthand in addition to explicit description of          │
│ important features.                                                                                             │
│                                                                                                                 │
│ ════════════════════════════════════════                                                                        │
│ MEMORY HIERARCHY                                                                                                │
│ ════════════════════════════════════════                                                                        │
│ Bank 0 (actions)   — sliding window, max 8 entries, compress warning at 6.                                      │
│                      Holds raw tagged events. Auto-evicts oldest when full.                                     │
│ Bank 1 (aggregate) — unbounded. Holds CHECKPOINT summaries of bank 0                                            │
│                      and a current [STATE\] entry.                                                              │
│ Higher banks       — unbounded. Created as needed for longer-term or                                            │
│                      cross-topic aggregation.                                                                   │
│                                                                                                                 │
│ Each bank always maintains:                                                                                     │
│   [STATE\]      — one entry reflecting the current known state at that level.                                   │
│   [CHECKPOINT\] — one or more compression summaries from the bank below.                                        │
│                                                                                                                 │
│ ════════════════════════════════════════                                                                        │
│ MEMORY RULES — follow exactly, every step                                                                       │
│ ════════════════════════════════════════                                                                        │
│                                                                                                                 │
│ 1. STORE NEW INFORMATION                                                                                        │
│    Always store all new information in bank 0 immediately with a descriptive tag.                               │
│    Format: memory_bank(mode='store', content='detail', query='TOPIC | TYPE | IMPORTANCE', bank=0)               │
│    TOPIC is a somewhat generic description of the subject of the event.                                         │
│    TYPE is one of: event, state, fact, opinion, goal                                                            │
│    IMPORTANCE is an integer between 0 and 10.                                                                   │
│                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  memory_bank(mode='search', query='San Francisco Los Angeles', bank=-1)                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: [bank=0(actions) | idx=0 | score=0] [CHESS GAME | EVENT | 5] White moves queen to h5
[bank=0(actions) | idx=1 | score=0] [STATE] White to move, Black king in check after White moves queen to h5
[bank=1(aggregate) | idx=0 | score=0] [STATE | CHESS | FACT | 9] Current state: Black king is in checkmate after 
the move Qxf7#
[bank=1(aggregate) | idx=1 | score=0] [CHECKPOINT | CHESS GAME | FACT | 9] White to move, Black king in check after
White moves pawn to d3. Previous moves include Black moving knight to f6, Black king moving to e7 to escape check, 
and Black rook capturing queen on g8, resulting in checkmate.
[bank=1(aggregate) | idx=2 | score=0] [CHECKPOINT | CHESS GAME | FACT | 9] Current state: White to move, Black king
in check after White moves pawn to d3. Previous moves include Black moving knight to f6, Black king moving to e7 to
escape check, and Black rook capturing queen on g8, resulting in checkmate.
[bank=1(aggregate) | idx=3 | score=0] [CHECKPOINT | CHESS GAME | FACT | 9] Current state: White to move, Black king
in check after White moves pawn to e4. Previous moves include Black moving pawn to e5.
[bank=1(aggregate) | idx=4 | score=0] [CHECKPOINT | CHESS GAME | FACT | 9] Current state: White to move, Black king
in check after White moves pawn to e4. Black moves knight to c6.
[bank=1(aggregate) | idx=5 | score=0] [STATE | CHESS | FACT | 9] Current state: White to move, Black king in check 
after White moves bishop to c4.
[bank=1(aggregate) | idx=6 | score=0] [CHESS MOVE | EVENT | 5] White moves bishop to c4 to defend against the check
on the Black king.
[bank=1(aggregate) | idx=7 | score=0] [STATE] Current state: Black king on e7 to escape check
[bank=1(aggregate) | idx=8 | score=0] [CHECKPOINT | CHESS GAME | FACT | 9] Current state: White to move, Black king
in check after White moves pawn to e4. Black moves pawn to d5 to continue the sequence leading to a check on the 
Black king.

[Step 1: Duration 2.54 seconds| Input tokens: 3,780 | Output tokens: 65]

...

Code parsing failed on line 17 due to: SyntaxError: unterminated string literal (detected at line 17) (<unknown>, 
line 17)
memory_bank(mode='store', content='The distance between Paris and Boston is approximately 5564 km. Travel          
^

[Step 17: Duration 39.31 seconds| Input tokens: 306,002 | Output tokens: 14,611]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 18 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Step 18: Duration 12.79 seconds]

KeyboardInterrupt: 

### Now for the city shapes:

1. The shape formed by connecting four cities as vertices on a map would typically form a quadrilateral, specifically a square if the cities are evenly spaced. However, the actual shape could vary depending on the specific locations of the cities.
2. BLANK
- (2's Judge) The provided answer does not match the spirit of the correct answer.

latency: 75.47811937332153 s

latency: 250.30946445465088 s

1. I need a description or a diagram of the shape along with the lengths of its edges to determine which connection forms the longest edge. (Also failed to verify if it matched the correct answer.)
2. Please provide details about the shape and its edges, including the lengths of each edge. (2's judge did not verify, but was able to calculate distance information for the other edges now!)

latency: 75.02980422973633 s

latency: 54.233598709106445 s

1. San Francisco, Los Angeles, Austin, Boston, and Tokyo form a roughly rectangular shape on a map.
- (1's Judge): The correct answer is an open pentagon shape — a chain of four edges with no closing line back to San Francisco. The given answer describes a rectangular shape formed by the cities.
2. The cities form a roughly quadrilateral shape, with San Francisco, Los Angeles, and Austin forming one side, and Boston and Tokyo forming the opposite side.
- (2's Judge) The answers do not match in spirit.

latency: 117.42378664016724 s

latency: 161.72940135002136 s


At this point on, since there was no clear delineation between the set of shapes for case 1 and 2, model 2 went into a loop and broke.
Interestingly, 2 always got the cities correct (via memory) and could easily book a trip given the web information, but could not really reason about the input.
Both still fail a lot -- on this subsample (did not do all the ones in #3 to save some time), both fail on all the questions.

# Part 4: Build a Multimodal Language Agent (30 points)

Empowering agents with mutlimodal capabilities is crucial for solving tasks that go beyond text processing. For instance, many real-world challenges, such as web browsing, automatic purchasing, document understanding, or robotics, require analyzing rich visual content. Fortunately, smolagents provides built-in support for vision-language models (VLMs), enabling agents to process and interpret images effectively.

See architecture below:
https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/smolagents-can-see/diagram_adding_vlms_smolagents.png

In this part, you will design and implement a multimodal language agent that solves a real multimodal task through multi-step interaction with an environment.

Your agent must use multimodal observations (such as images/screenshots/egocentric data, audio or other non-text modalities) as part of its decision-making, not only text prompts.

#### Problem 1: Vision Implementation and Controlled Comparison (20 points)

Build a vision-enhanced version of your previous agent.

Minimum requirements (also your deliverables):
- Start from your Part 3 agent and add multimodal observations into the decision loop. If multimodality is irrelevant to your task, for this exercise propose a new task (examples: web navigation, chart QA, UI automation, visual fact-checking, document understanding, image-based shopping assistant, map/screenshot reasoning, robot interaction).
- Implement an agent that consumes multimodal data at each step (e.g. image/screenshot input like in the template example below).
- Preserve step-level logs so we can inspect observation -> action -> result.
- Run a controlled comparison on the same task set:
  - Version A: text-only baseline
  - Version B: vision-enhanced agent (this section)

Required metrics:
- 1 architecture diagram
- Comparison of task success rate for Version A vs Version B
- 2 qualitative trace examples (one success, one failure)
- 1 short discussion of trade-offs
---

**NOTE: YOU MAY HAVE TO DOWNLOAD THIS NOTEBOOK AND RUN IT LOCALLY TO MAKE THE EXAMPLE CODE WORK**. The example code controls your chrome browser which google colab does not have.

In [ ]:
from smolagents import TransformersModel
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers import StoppingCriteria, StoppingCriteriaList
import torch

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for local inference in this notebook.')

# Load model and processor once
# _vl_processor = AutoProcessor.from_pretrained(MODEL_ID)
# _vl_model = AutoModelForImageTextToText.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )
# _vl_model.eval()

vl_model = TransformersModel(
    model_id=MODEL_ID,
    # model=_vl_model,
    # tokenizer=_vl_processor.tokenizer,
    # processor=_vl_processor,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print('Model ready:', MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model ready: Qwen/Qwen2.5-VL-7B-Instruct


In [ ]:
# del vl_model
# del _vl_processor
# del _vl_model
# del agent
# import gc
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
!pip install selenium helium pillow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 14.3 MB/s eta 0:00:00


In [ ]:
!pip install webdriver_manager selenium helium pillow -q

import sys
import os
from webdriver_manager.chrome import ChromeDriverManager

!apt-get update
!apt-get remove --purge chromium-browser chromium-chromedriver -y # Remove existing to avoid conflicts
!apt-get autoremove -y

# Install google-chrome-stable
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable -y

# Use webdriver_manager to install and get chromedriver path
CHROME_DRIVER_PATH = ChromeDriverManager().install()
os.environ['CHROME_DRIVER_PATH'] = CHROME_DRIVER_PATH

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,642 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Hit:13 https://ppa.launchpadcontent.net/graphi

**Action:** In this example, we will create a set of agent tools specifically designed for browsing, such as `search_item_ctrl_f`, `go_back`, and `close_popups`. These tools allow the agent to act like a person navigating the web. Note: It is possible to extend these with other navigation tools such as `mouse_click` and `keypress`.

In [ ]:
from io import BytesIO
from time import sleep

import helium
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from smolagents import CodeAgent, Tool
from smolagents.agents import ActionStep


class SearchItemCtrlFTool(Tool):
    name = "search_item_ctrl_f"
    description = "Searches for text on the current page and jumps to the nth occurrence."
    inputs = {
        "text": {"type": "string", "description": "The text to search for"},
        "nth_result": {"type": "integer", "description": "Which occurrence to jump to", "nullable": True},
    }
    output_type = "string"

    def forward(self, text: str, nth_result: int = 1) -> str:
        driver = helium.get_driver()
        elements = driver.find_elements(By.XPATH, f"//*[contains(text(), '{text}')]")
        if nth_result > len(elements):
            raise Exception(f"Match n°{nth_result} not found (only {len(elements)} matches found)")
        result = f"Found {len(elements)} matches for '{text}'."
        elem = elements[nth_result - 1]
        driver.execute_script("arguments[0].scrollIntoView(true);", elem)
        result += f"Focused on element {nth_result} of {len(elements)}"
        return result

class GoBackTool(Tool):
    name = "go_back"
    description = "Goes back to previous page."
    inputs = {}
    output_type = "string"

    def forward(self) -> str:
        driver = helium.get_driver()
        driver.back()
        return "Went back to the previous page."

class ClosePopupsTool(Tool):
    name = "close_popups"
    description = "Closes visible modal or pop-up windows with ESC. This does not work on cookie consent banners."
    inputs = {}
    output_type = "string"

    def forward(self) -> str:
        driver = helium.get_driver()
        webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()
        return "Sent ESC to close popups (if any)."

**Perception:** In this example, we also need functionality for saving screenshots, as this will be an essential part of what our VLM agent uses to complete the task. This functionality captures the screenshot and saves it in step_log.observations_images = [image.copy()], allowing the agent to store and process the images dynamically as it navigates.

In [ ]:
from PIL import Image
import requests
from io import BytesIO
from time import sleep

import helium
from selenium import webdriver
from smolagents import ActionStep, CodeAgent

# Configure Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--force-device-scale-factor=1")
chrome_options.add_argument("--window-size=1000,1350")
chrome_options.add_argument("--disable-pdf-viewer")
chrome_options.add_argument("--window-position=0,0")
# Added necessary options for Colab
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Initialize the browser with headless=True for Colab
driver = helium.start_chrome(headless=True, options=chrome_options)

def save_screenshot(step_log: ActionStep, agent: CodeAgent) -> None:
    sleep(1.0)  # Let JavaScript animations happen before taking the screenshot
    driver = helium.get_driver()
    current_step = step_log.step_number
    if driver is not None:
        prior_steps = getattr(getattr(agent, "memory", None), "steps", [])
        for step_logs in prior_steps:  # Remove previous screenshots from logs for lean processing
            if isinstance(step_logs, ActionStep) and step_logs.step_number <= current_step - 2:
                step_logs.observations_images = None
        png_bytes = driver.get_screenshot_as_png()
        image = Image.open(BytesIO(png_bytes))
        print(f"Captured a browser screenshot: {image.size} pixels")
        step_log.observations_images = [image.copy()]  # Create a copy to ensure it persists, important!
    else:
      driver = helium.start_chrome(headless=True, options=chrome_options)

    # Update observations with current URL
    url_info = f"Current url: {driver.current_url}"
    step_log.observations = url_info if step_log.observations is None else step_log.observations + "\n" + url_info
    return

In [ ]:
import os

# Fix for SessionNotCreatedException in Colab
# If the previous cell ran, this should now work.
# Re-initialize the driver, ensuring it's headless and with necessary options.

# Ensure the helium driver is properly reset or reinitialized
# If a driver was already started and failed, it might be in a bad state.
# We'll explicitly stop any existing driver if it was started in the previous failed attempt.
try:
    if helium.get_driver() is not None:
        helium.kill_browser()
except:
    pass # No active driver to kill

# Configure Chrome options again (essential for headless mode in Colab)
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless=new") # Newer headless mode
chrome_options.add_argument("--no-sandbox") # Required for Colab
chrome_options.add_argument("--disable-dev-shm-usage") # Required for Colab
chrome_options.add_argument("--disable-gpu") # Often needed for headless in Colab
chrome_options.add_argument("--window-size=1920,1080") # Set a reasonable window size
chrome_options.add_argument("--force-device-scale-factor=1")
chrome_options.add_argument("--window-position=0,0")
chrome_options.add_argument("--disable-pdf-viewer")

# Specify the path to the Chrome binary
chrome_options.binary_location = "/usr/bin/google-chrome" # Path to google-chrome-stable

# Get chromedriver path from environment variable set in previous cell
CHROME_DRIVER_PATH = os.environ.get('CHROME_DRIVER_PATH')
if not CHROME_DRIVER_PATH:
    raise RuntimeError("CHROME_DRIVER_PATH not set. Run the previous installation cell first.")

# Start the browser with headless=True and the specified executable_path
driver = helium.start_chrome(headless=True, options=chrome_options)#, executable_path=CHROME_DRIVER_PATH)

print("Helium Chrome driver initialized successfully.")

Helium Chrome driver initialized successfully.


This function is passed to the agent as step_callback, as it’s triggered at the end of each step during the agent’s execution. This allows the agent to dynamically capture and store screenshots throughout its process.

Now, we can generate our vision agent for browsing the web, providing it with the tools we created. This tool will help the agent retrieve necessary information for verifying guests’ identities based on visual cues.

In [ ]:
helium_instructions = """
You can use helium to access websites. Don't bother about the helium driver, it's already managed.
We've already ran "from helium import *"
Then you can go to pages!
Code:
```py
go_to('github.com/trending')
```<end_code>

You can directly click clickable elements by inputting the text that appears on them.
Code:
```py
click("Top products")
```<end_code>

If it's a link:
Code:
```py
click(Link("Top products"))
```<end_code>

If you try to interact with an element and it's not found, you'll get a LookupError.
In general stop your action after each button click to see what happens on your screenshot.
Never try to login in a page.

To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.
Code:
```py
scroll_down(num_pixels=1200) # This will scroll one viewport down
```<end_code>

When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or targeting an 'X' element (this most often fails).
Just use your built-in tool `close_popups` to close them:
Code:
```py
close_popups()
```<end_code>

You can use .exists() to check for the existence of an element. For example:
Code:
```py
if Text('Accept cookies?').exists():
    click('I accept')
```<end_code>

# IMPORTANT
When outputting code output it in <code> tags.

# ONLY USE AVAILABLE TOOLS!
"""

In [ ]:
import os

from smolagents import CodeAgent, OpenAIServerModel

# # ============================================================
# # ############ IF YOU CANT RUN THE MODEL LOCALLY #############
# # ============================================================
# if os.getenv("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY").strip()

# model = OpenAIServerModel(model_id="gpt-5.4-mini")
# # ============================================================
# # ########## END IF YOU CANT RUN THE MODEL LOCALLY ###########
# # ============================================================

web_search='go_to'
SYSTEM_INSTRUCTIONS = f"""
Do not assume anything; ask all questions necessary to determine the intent of any query.
When required, cite sources as URLs. If uncertain, say so explicitly.
Always store all new state information in bank 0 immediately with a descriptive tag.
Do this prior to any reasoning or information compression.

When compressing information, feel free to use common shorthand in addition to explicit description of important features.
════════════════════════════════════════
BROWSING AND VISUAL REASONING
════════════════════════════════════════
- When asked to find specific information on a webpage, first navigate using go_to().
- Use search_item_ctrl_f with ONE keyword at a time to locate information.

TOOL RETURN VALUES — READ THIS CAREFULLY:
- search_item_ctrl_f returns ONLY a plain string like "Found 10 matches. Focused on element 1 of 10."
- It does NOT return a Python object. There is no search_result variable.
- There is no .highlighted_text, .sentence, .text, or any attribute to access.
- Do NOT write: search_result[0], result.text, matches[0].sentence — these will always fail.

HOW TO READ THE ANSWER:
- After search_item_ctrl_f runs, a screenshot image is injected into your context.
- The screenshot shows the browser with the matched word highlighted on screen.
- YOUR ONLY JOB is to look at the screenshot image and read the sentence around the highlight.
- Then call final_answer() with exactly what you read.

VALID NEXT ACTION after a successful search_item_ctrl_f:
  final_answer("The sentence you read from the screenshot containing the word.")

INVALID NEXT ACTIONS — never do these:
  search_result[0].highlighted_text  ← no such variable
  search_item_ctrl_f("ABC")         ← do not repeat the same search
  print(anything)                    ← printing won't show you the page content

The screenshot is your eyes. Read it. Answer. Done.

════════════════════════════════════════
MEMORY HIERARCHY
════════════════════════════════════════
Bank 0 (actions)   — sliding window, max 8 entries, compress warning at 6.
                     Holds raw tagged events. Auto-evicts oldest when full.
Bank 1 (aggregate) — unbounded. Holds CHECKPOINT summaries of bank 0
                     and a current [STATE] entry.
Higher banks       — unbounded. Created as needed for longer-term or
                     cross-topic aggregation.

Each bank always maintains:
  [STATE]      — one entry reflecting the current known state at that level.
  [CHECKPOINT] — one or more compression summaries from the bank below.

════════════════════════════════════════
MEMORY RULES — follow exactly, every step
════════════════════════════════════════

1. STORE NEW INFORMATION
   Always store all new information in bank 0 immediately with a descriptive tag.
   Format: memory_bank(mode='store', content='detail', query='TOPIC | TYPE | IMPORTANCE', bank=0)
   TOPIC is a somewhat generic description of the subject of the event.
   TYPE is one of: event, state, fact, opinion, goal
   IMPORTANCE is an integer between 0 and 10.

2. UPDATE CURRENT STATE
   After every store, update the [STATE] entry in that bank:
     a. Remove the old state:
        memory_bank(mode='remove', query='STATE', bank=0)
     b. Store the new state:
        memory_bank(mode='store', content='<concise current state>', query='STATE', bank=0)
   Do the same for bank 1 after any compression.

3. COMPRESS ON WARNING
   When you receive a ⚠ COMPRESS SOON warning:
     a. Immediately call compress BEFORE any further stores:
        memory_bank(mode='compress', bank=0)
     b. Read the returned COMPRESS_TASK carefully.
     c. Write a highly detailed summary that describes the entire system state.
     d. Store the checkpoint in bank 1:
        memory_bank(mode='store', content='<your summary>', query='CHECKPOINT | TOPIC | TYPE | IMPORTANCE', bank=1)
     e. Update [STATE] in bank 1 to reflect the new aggregate state.
   Bank 0 entries are cleared after compression, except a small overlap which remains until auto-evicted.
   The overlap ensures the last 2 entries of each window appear in two consecutive checkpoints.

4. AUTO-EVICTION
   When bank 0 hits max_size=8, the oldest entries are auto-evicted automatically.
   You will receive a ⚠ BANK FULL warning listing what was evicted.
   If those entries were not yet compressed, that data is permanently lost.
   Always compress at the ⚠ COMPRESS SOON warning (step 3) to prevent this.

5. ANSWER QUESTIONS FROM MEMORY FIRST
   Before using {web_search} or any reasoning, always search memory:
     memory_bank(mode='search', query='<relevant keywords>', bank=-1)
   bank=-1 searches all banks simultaneously.
   Only call {web_search} if search returns "no memory" or "no matches".
   When storing many small points of related web information in bank 0, use a single memory_bank store call with all the information together.

6. RETRIEVAL STRATEGY
   - For recent events      → search bank 0 first, then bank=-1 if needed.
   - For historical/summary → search bank 1+ first, then bank=-1 if needed.
   - For current state      → memory_bank(mode='search', query='STATE', bank=-1)
   - If unsure what exists  → memory_bank(mode='list_banks') then search.

7. ACCURACY
   If a past action is impossible or incorrect, stop and report it immediately.
   False information must be identified and corrected before proceeding.
   Cross-reference bank 1 checkpoints against raw bank 0 entries when accuracy is critical.

════════════════════════════════════════
OUTPUT RULES
════════════════════════════════════════
- The user does not always expect a response. Do NOT provide one unless explicitly asked.
- If no action is needed and no question was asked, terminate with:
  <code>
  final_answer("")
  </code>
- Do not write any text outside a code block as your final output.
- Never call {web_search} before searching memory (bank=-1) first.
"""

# """
# """

topic_router = DynamicTopicRouter()
shared_memory = MemoryBank(topic_router)

# Create the agent
agent = CodeAgent(
    tools=[GoBackTool(), ClosePopupsTool(), SearchItemCtrlFTool()],
    model=vl_model,
    additional_authorized_imports=["helium"],
    step_callbacks=[save_screenshot],
    max_steps=20,
    verbosity_level=2,
)

# Import helium for the agent
agent.python_executor("from helium import *")

CodeOutput(output=None, logs='', is_final_answer=False)

In [ ]:
search_request = """
Please navigate to https://en.wikipedia.org/wiki/Chicago and give me a sentence containing the word "1992" that mentions a construction accident.
"""

agent_output = agent.run(helium_instructions + search_request)
print("Final output:")
print(agent_output)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You can use helium to access websites. Don't bother about the helium driver, it's already managed.              │
│ We've already ran "from helium import *"                                                                        │
│ Then you can go to pages!                                                                                       │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ go_to('github.com/trending')                                                                                    │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ You can directly click clickable elements by inputting the text that appears on them.                           │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ click("Top products")                                                                                           │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ If it's a link:                                                                                                 │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ click(Link("Top products"))                                                                                     │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ If you try to interact with an element and it's not found, you'll get a LookupError.                            │
│ In general stop your action after each button click to see what happens on your screenshot.                     │
│ Never try to login in a page.                                                                                   │
│                                                                                                                 │
│ To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.     │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ scroll_down(num_pixels=1200) # This will scroll one viewport down                                               │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or   │
│ targeting an 'X' element (this most often fails).                                                               │
│ Just use your built-in tool `close_popups` to close th

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To achieve this, I need to navigate to the Wikipedia page for Chicago and look for a sentence containing  
the word "1992" that mentions a construction accident. I will use the `go_to` function to navigate to the page and 
then use the `search_item_ctrl_f` function to find the specific sentence.                                          
                                                                                                                   
<code>                                                                                                             
go_to('https://en.wikipedia.org/wiki/Chicago')                                                                     
search_item_ctrl_f(text="1992", nth_result=1)                                                                      
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  go_to('https://en.wikipedia.org/wiki/Chicago')                                                                   
  search_item_ctrl_f(text="1992", nth_result=1)                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: Found 4 matches for '1992'.Focused on element 1 of 4

...

Out: None

Captured a browser screenshot: (1920, 941) pixels


[Step 4: Duration 4.47 seconds| Input tokens: 22,909 | Output tokens: 412]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

MaxRetryError: HTTPConnectionPool(host='localhost', port=51879): Max retries exceeded with url: /session/abc8eb58c48f29addb084342ee2ca810/screenshot (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7dc8cf59c950>: Failed to establish a new connection: [Errno 111] Connection refused'))

#### Problem 2: Safety and Policy Evaluation (10 points)

Add a targeted safety/policy evaluation for your agent and compare behavior before and after one mitigation.

Minimum requirements (also your deliverables):
- Design at least **3** challenging prompts relevant to your domain (for example: unsafe requests, privacy-sensitive requests, or out-of-scope requests).
- Define expected safe behavior for each prompt before running the test.
- Run your agent on all prompts and record observed behavior.
- Implement at least one mitigation (for example: improved system instruction, tool guardrail, or explicit refusal policy) and re-run the same prompts.
- Report one table with **before/after** behavior and a short reflection on trade-offs (false refusals vs missed refusals).

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

from smolagents import CodeAgent, VisitWebpageTool, WebSearchTool
import datetime


SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI 2026 course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
    'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is. {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

test_query = "What is a project one current student is working on? Find one and summarize it."
print(agent.run(f"{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}"))

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

# Part 5: Agent Observability and Evaluation (20 points)

So far, we have done offline evaluation of our model on a subset of our dataset with ground truth labels. In this part, we will explore observability and online evaluation of our agent. In the following demonstration, we will use Langfuse as our observability tool, but you can use any other OpenTelemetry-compatible services (like TruLens). The code below shows how to set environment variables for Langfuse (or any OTel endpoint) and how to instrument your smolagent.

#### Problem 1: Setting up an Observability Monitor (5 points)

Set up an observability backend (Langfuse or any OpenTelemetry-compatible service) so each agent run is traceable end-to-end.

In [ ]:
# STEP 1: Install dependencies
!pip install -q langfuse 'smolagents[telemetry]' openinference-instrumentation-smolagents datasets 'smolagents[gradio]' gradio --upgrade

In [ ]:
import os
from getpass import getpass

# Get keys for your project from the project settings page: https://us.cloud.langfuse.com
# Use environment variables only; do not hardcode secrets in notebook source.
if not os.getenv("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass("Enter LANGFUSE_SECRET_KEY: ")

if not os.getenv("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("Enter LANGFUSE_PUBLIC_KEY: ")

# You can override this if you use self-hosted Langfuse.
os.environ.setdefault("LANGFUSE_BASE_URL", "https://us.cloud.langfuse.com")

print("Langfuse environment variables configured.")

## Remember to remove the keys for before your submission!


In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

#### Problem 2: Record and Inspect Traces (5 points)

Next, set up SmolagentsInstrumentor() to instrument your smolagent and send traces to Langfuse. Then run your Part 3/4 agent and inspect trace behavior in the dashboard.

Minimum requirements:
- Show evidence that at least 5 runs were recorded as traces.
- For at least 2 runs, inspect spans and identify: model call count, tool call sequence, and where most latency occurred.
- Include at least 1 run where behavior was incorrect or suboptimal, and diagnose whether the issue came from reasoning, tool output, prompt/instructions, or infrastructure.
- Link each diagnosis to specific trace evidence (span names, timing, tool output, or error text).

In [ ]:
!pip install markdownify requests

In [ ]:
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
import datetime
SmolagentsInstrumentor().instrument()

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

from smolagents import CodeAgent, VisitWebpageTool, WebSearchTool

course_agent = CodeAgent(
    tools=[VisitWebpageTool(), WebSearchTool()],
    model=model,
)

SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI 2026 course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
    'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is. {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

test_query = "What are the current student projects?"
print(course_agent.run(f"{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}"))

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

Check your [Langfuse Traces Dashboard](https://cloud.langfuse.com/) (or your chosen observability tool) to confirm that the spans and logs have been recorded.

#### Problem 3: Online Evaluation (10 points)

In a previous section, we learned about the difference between observability, online, and offline evaluation. Now, we will monitor your agent under live-like conditions and evaluate trade-offs across configuration choices.

Read more: [Monitoring and evaluating agents](https://huggingface.co/learn/agents-course/en/bonus-unit2/monitoring-and-evaluating-agents-notebook).

Common metrics include:
- Costs: token usage, which you can transform into approximate costs by assigning a price per token.
- Latency: time it takes to complete each step, or the entire run.
- User feedback: in real-life deployment, users can often provide direct feedback to help refine or correct the agent (such as thumbs up or down with explanation).
- LLM-as-a-judge: use a separate LLM to evaluate your agent's output in near real-time (e.g., checking for toxicity, correct tool use, user response quality, or correctness).

Minimum requirements:
- Change at least two parameters of your agent such as the LLM model, planning steps, tool set size, or memory architecture (for inspiration see the [smolagents documentation](https://huggingface.co/docs/smolagents/)).
- Evaluate each configuration on the same set of at least 5 prompts.
- Track at least 3 metrics per configuration (for example success rate, average latency, and estimated cost).
- Attach screenshots of relevant Langfuse results in your hand-in.

Deliverables:
- One comparison table with each configuration and all reported metrics.
- A short discussion (6-8 sentences): how your parameter changes impacted results, where trade-offs appeared, and which setup you would deploy. Consider how user feedback or LLM-as-a-judge could be integrated in future online evaluations.

In [ ]:
# You can make write your agents with different architectural changes here:


# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

from smolagents import CodeAgent, VisitWebpageTool, WebSearchTool

course_agent = CodeAgent(
    tools=[GetCurrentStudentProjectsTool(), VisitWebpageTool(), WebSearchTool()],
    model=model,
)

SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
)

test_query = "What are the current student projects?"
print(course_agent.run(f"{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}"))

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

# Part 6: Integrate Your Agent into Our MMAI Agent Discord World (10 points)

You will now integrate an agent into our Discord world. Feel free to use the agent from the previous sections or build an entirely new one. Fun agents are encouraged! After class, we will run all agents at the same time to have them exist together in Discord.

To get started, please join the server using [this link](https://discord.gg/DEzs78ud).

1. Go to https://discord.com/developers/applications/ and click New Application.
2. Open the 'Bot' tab.
3. Set icon (this will be the profile image in Discord) and username.
4. Generate a token and save it. This will be used in the code below.
5. Enable 'Public Bot', 'Presence Intent', 'Server Members Intent', and 'Message Content Intent'.

In [ ]:
!pip install -q -U discord.py
!curl -L "https://github.com/valleballe/mmai/blob/master/static/utils.py?raw=1" -o utils.py

import os
from getpass import getpass

# Set your DISCORD_TOKEN securely in environment variables before running.
if not os.getenv("DISCORD_TOKEN"):
    os.environ["DISCORD_TOKEN"] = getpass("Enter DISCORD_TOKEN: ")

DISCORD_TOKEN = os.getenv("DISCORD_TOKEN")
print("DISCORD_TOKEN loaded from environment.")

Feel free to paste and edit your agent from prior parts of the notebook below.

In [ ]:
# Use your custom course tool from earlier in this notebook.
# This assumes GetCurrentStudentProjectsTool is already defined in Part 3.

from smolagents import CodeAgent, VisitWebpageTool, WebSearchTool
import datetime

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI 2026 course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
    'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is. {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

course_agent = CodeAgent(
    tools=[GetCurrentStudentProjectsTool(), VisitWebpageTool(), WebSearchTool()],
    model=model,
)

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

Next, we will run the agent so that it is accessible on the Discord. You will be able to interact with the agent while the cell below is running. Feel free to play around with what triggers the agent: maybe the agent responds to every single message,  or maybe it only responds when tagged (as in current implementation), or maybe it gets triggered by specific words. Also consider that it could trigger other bots by @tagging them.

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

import asyncio
import discord
from discord.ext import commands
from utils import _hydrate_user_mentions


# Discord bot setup
intents = discord.Intents.default()
intents.message_content = True
intents.members = True
bot = commands.Bot(command_prefix="!", intents=intents)

@bot.event
async def on_ready():
    print(f"{bot.user} is online and listening for @mentions.")

@bot.event
async def on_message(message):
    # This triggers whenever someone sends a message in a channel
    if message.author.bot:
        return

    # Check if the agent is mentioned in the message and respond with the agent's answer
    if bot.user and bot.user.mentioned_in(message):
        user_prompt = (
            message.content
            .replace(f"<@{bot.user.id}>", "")
            .replace(f"<@!{bot.user.id}>", "")
            .strip()
        )

        # If the agent is mentioned without a question, prompt the user to ask a question
        if not user_prompt:
            await message.channel.send(
                "Mention me with a question, e.g. @BotName what are current student projects?"
            )
        else:
            full_prompt = f"{SYSTEM_INSTRUCTIONS}\n\nUser question: {user_prompt}"
            async with message.channel.typing():
                response = await asyncio.to_thread(course_agent.run, full_prompt)
                response_text = _hydrate_user_mentions(str(response), message.guild, message.author)

                if len(response_text) > 2000:
                    response_text = response_text[:1997] + "..."

                await message.channel.send(
                    response_text,
                    allowed_mentions=discord.AllowedMentions(users=True, roles=False, everyone=False),
                )

    await bot.process_commands(message)

bot.run(DISCORD_TOKEN)

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

To add the agent to the Discord server:
1. Open OAuth2.
2. Enable 'bot' and 'applications.commands'.
3. Under bot permissions, enable 'Send Messages', 'Embed Links', and 'Read message history'. You may need additional permissions depending on your specific needs.

Under 'Generated URL', copy-paste the URL into your browser. This should prompt you to add your agent to a server. Please add it to 'MMAI Agents World'. If you do not see the server, please join it using [this link](https://discord.gg/DEzs78ud).

Reflection and documentation (required):
Write 4-6 sentences reflecting on trigger strategy for your bot. For example, compare always-on response, @mention-only response (this implementation), keyword-triggered response, or letting the LLM decide whether to respond. Include documentation of Discord interactions with the bot in your write-up.

## Optional (10 points): Try OpenClaw

In this optional exercise, experiment with [OpenClaw](https://openclaw.ai/) to explore a more production-style, multi-channel agent system. Set it up locally or via the provided quickstart, connect it to a simple environment (e.g., WhatsApp, Slack, Discord, or CLI), and try building or using at least one “skill” or agent behavior. Submit 2–3 screenshots demonstrating your interaction (e.g., task execution, tool/skill use, or multi-step behavior). In a short reflection (1–2 paragraphs), compare this experience to your smolagents implementation: how does the architecture differ (e.g., abstraction layers, persistence, skills/plugins, orchestration)? Why do you think systems like OpenClaw have become popular? What risks or failure modes emerge when agents are persistent, extensible, and connected to external tools? And lastly, how do you foresee LLM agents developing in the next 5-10 years?